# OntoKG-EQ — GraphRAG/LLM explanation-faithfulness (free GPU)

**Before running:** Settings → Accelerator = **GPU T4 x2**, Internet = **On**, and **Add Input** the dataset containing the four `*.ttl` files (`demo_psx_inferred.ttl`, `demo_msx_inferred.ttl`, `demo_idx_inferred.ttl`, and **`demo_idx_scaled.ttl`**).

Cell 1 installs dependencies; Cell 2 runs the experiment with a local open model (no API key). It now scores **two cohorts**: the curated worked cases (base) **and the scaled 64-stock Indonesia cohort (~41 cases: 24 CQ3 + 17 CQ1)**. Copy the `GraphRAG/LLM (scaled)` row into the paper.


In [ ]:
!pip -q install rdflib transformers accelerate bitsandbytes sentencepiece

In [ ]:
import os, re, glob, csv
from rdflib import Graph, RDF, URIRef
from rdflib.namespace import Namespace

CORE = Namespace("https://w3id.org/ontokg-eq#")
MODEL = "Qwen/Qwen2.5-7B-Instruct"   # FULL id (org/name). Ungated picks: Qwen/Qwen2.5-3B-Instruct, microsoft/Phi-3.5-mini-instruct, HuggingFaceTB/SmolLM2-1.7B-Instruct
OUT   = "/kaggle/working"

def _find(name, default="/kaggle/input"):
    h = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return h[0] if h else None

BASE_DIR   = os.path.dirname(_find("demo_idx_inferred.ttl") or "/kaggle/input/x")
SCALED_TTL = _find("demo_idx_scaled.ttl")
print("base dir:", BASE_DIR, "| scaled ttl:", SCALED_TTL)

def short(x): return str(x).split("#")[-1] if x else ""
PRED = re.compile(r"\b(because|due to|driven by|will|likely|expected to|forecast|predict|target|recommend|buy|sell|cause|outlook|going to|should)\b", re.I)
NUM  = re.compile(r"-?\d+\.\d+")

# ---- curated worked cases (base: AnalyticalFinding nodes in the 3 inferred graphs) ----
def build_cases():
    cases=[]
    for m in ["psx","msx","idx"]:
        fp=os.path.join(BASE_DIR,f"demo_{m}_inferred.ttl")
        if not os.path.exists(fp): continue
        g=Graph().parse(fp,format="turtle")
        for f in g.subjects(RDF.type, CORE.AnalyticalFinding):
            ft=str(next(g.objects(f,CORE.hasFindingType),""))
            if ft not in ("relative-outperformer","fundamentals-market-divergence"): continue
            ent=next(g.objects(f,CORE.concernsEntity),None)
            label=str(next(g.objects(ent,CORE.hasCompanyName),short(ent)))
            b=URIRef(str(f)+"_bundle"); atoms=[]; nums=[]
            for o in g.objects(b,CORE.includesObservation):
                mn=str(next(g.objects(o,CORE.hasMetricName),"")); v=next(g.objects(o,CORE.hasMetricValue),None)
                tgt=short(next(g.objects(o,CORE.isObservationOf),None))
                if v is not None:
                    fv=round(float(v),2); atoms.append(f"{tgt} {mn} = {fv}"); nums.append(fv)
            ev=next(g.objects(b,CORE.containsEvidenceItem),None)
            date=str(next(g.objects(ev,CORE.hasAnnouncementDate),"")) if ev else ""
            src=next(g.objects(ev,CORE.hasEvidenceSource),None) if ev else None
            stype=str(next(g.objects(src,CORE.hasSourceType),"")) if src else ""
            ctx="Facts present in the validated knowledge graph:\n - "+"\n - ".join(atoms)
            if date: ctx+=f"\n - evidence: {label} FY results announced {date}"
            if src:  ctx+=f"\n - source: {short(src)} ({stype})"
            q=(f"Did {label} outperform both its sector and the broad-market benchmark over the post-report window? Explain, citing the evidence."
               if ft=="relative-outperformer" else
               f"Did {label} show stronger fundamentals but a weaker market response than the benchmark? Explain, citing the evidence.")
            onto=f"{label}: "+"; ".join(atoms)+". "+(f"Evidence: FY results announced {date} (source: {short(src)})." if src else "")
            cases.append(dict(market=m,label=label,ftype=ft,question=q,context=ctx,
                              factnums=set(round(n,2) for n in nums),
                              prov_tokens=[date, short(src) if src else "", "disclosure","Stock Exchange"],
                              onto_answer=onto))
    return cases

# ---- scaled cohort (built directly from derived metrics; CQ1/CQ3 SPARQL is engine-bound at 37k triples) ----
def build_scaled_cases(ttl):
    if not ttl or not os.path.exists(ttl): return []
    g=Graph().parse(ttl,format="turtle")
    post={}; bench={}; sect={}; growth={}
    for o in g.subjects(CORE.hasMetricName,None):
        mn=str(next(g.objects(o,CORE.hasMetricName))); v=next(g.objects(o,CORE.hasMetricValue),None)
        if v is None: continue
        val=float(v); ent=next(g.objects(o,CORE.isObservationOf),None); w=next(g.objects(o,CORE.observedOverWindow),None)
        if   mn=="post-report window return %" and w is not None: post[w]=(ent,val)
        elif mn=="benchmark window return %"  and w is not None: bench[w]=val
        elif mn=="sector window return %"     and w is not None: sect[w]=val
        elif mn=="YoY profit growth %": growth[ent]=val
    ann={}
    for a in g.subjects(RDF.type, CORE.Announcement):
        e=next(g.objects(a,CORE.aboutCompany),None); d=str(next(g.objects(a,CORE.hasAnnouncementDate),""))
        s=next(g.objects(a,CORE.hasEvidenceSource),None)
        ann[e]=(d, short(s) if s else "", str(next(g.objects(s,CORE.hasSourceType),"")) if s else "")
    cases=[]
    for w,(ent,cret) in post.items():
        br=bench.get(w); sr=sect.get(w); gr=growth.get(ent)
        label=str(next(g.objects(ent,CORE.hasCompanyName),short(ent)))
        d,src,st=ann.get(ent,("","",""))
        prov=[d,src,"Yahoo","Indonesia Stock Exchange","disclosure"]
        if sr is not None and br is not None and cret>sr and cret>br:
            atoms=[f"{label} return = {round(cret,2)}", f"sector return = {round(sr,2)}", f"benchmark return = {round(br,2)}"]
            cases.append(dict(market="idx_scaled",label=label,ftype="relative-outperformer",
                question=f"Did {label} outperform both its sector and the broad-market benchmark over the window? Explain, citing the source.",
                context="Facts present in the validated knowledge graph:\n - "+"\n - ".join(atoms)+(f"\n - evidence: FY results announced {d} (source: {src})" if d else ""),
                factnums={round(cret,2),round(sr,2),round(br,2)}, prov_tokens=prov,
                onto_answer=f"{label}: "+"; ".join(atoms)+f". Evidence: FY results announced {d} (source: {src})."))
        if gr is not None and br is not None and gr>0 and cret<br:
            atoms=[f"{label} YoY profit growth = {round(gr,2)}", f"{label} return = {round(cret,2)}", f"benchmark return = {round(br,2)}"]
            cases.append(dict(market="idx_scaled",label=label,ftype="fundamentals-market-divergence",
                question=f"Did {label} report stronger fundamentals but a weaker market response than the benchmark? Explain, citing the source.",
                context="Facts present in the validated knowledge graph:\n - "+"\n - ".join(atoms)+(f"\n - evidence: FY results announced {d} (source: {src})" if d else ""),
                factnums={round(gr,2),round(cret,2),round(br,2)}, prov_tokens=prov,
                onto_answer=f"{label}: "+"; ".join(atoms)+f". Evidence: FY results announced {d} (source: {src})."))
    return cases

def score(ans,factnums,prov,tol=0.05):
    ns=[round(float(x),2) for x in NUM.findall(ans)]
    mt=[n for n in ns if any(abs(n-t)<=tol for t in factnums)]
    return {"numbers":len(ns),"num_faithful":round(len(mt)/len(ns),2) if ns else 1.0,
            "halluc":len(ns)-len(mt),"unsupported":len(PRED.findall(ans)),
            "provenance":1 if any(t and str(t).lower() in ans.lower() for t in prov) else 0}

# ---- load the open-weight model (local; no API key) ----
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduce fragmentation
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")  # only for gated models

# free any model left in VRAM by a previous run in THIS kernel (prevents CUDA OOM when switching models)
for _v in ("pipe", "mdl"):
    if _v in globals():
        try: del globals()[_v]
        except Exception: pass
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)

def _load_4bit(device_map, max_memory=None):
    cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(
        MODEL, quantization_config=cfg, device_map=device_map, max_memory=max_memory,
        low_cpu_mem_usage=True, token=HF_TOKEN)

# A 4-bit 14B is ~8 GB and fits ONE T4. device_map="auto" can instead shard it in fp16 across both GPUs
# and OOM, so we pin the whole 4-bit model to a single GPU first; CPU offload is the last-resort safety net.
_attempts = [
    ("GPU0",            dict(device_map={"": 0})),
    ("GPU1",            dict(device_map={"": 1})),
    ("sharded+CPU off", dict(device_map="auto", max_memory={0: "13GiB", 1: "13GiB", "cpu": "40GiB"})),
]
mdl = None; _last = None
for _name, _kw in _attempts:
    try:
        gc.collect(); torch.cuda.empty_cache()
        mdl = _load_4bit(**_kw); print("loaded 4-bit via:", _name); break
    except Exception as _e:
        _last = _e; print(f"attempt [{_name}] failed:", repr(_e)[:140])
if mdl is None:
    raise RuntimeError(
        "All 4-bit attempts failed. Do Run -> Restart Session, then Run All with THIS MODEL only. "
        "If it still fails, this model is too big for the free GPU — the two-model result already in the "
        f"paper (Qwen2.5-7B + SmolLM2-1.7B) is sufficient. Last error: {repr(_last)[:140]}")
pipe = pipeline("text-generation", model=mdl, tokenizer=tok)

def llm_answer(q,ctx):
    prompt=(f"You are a financial analysis assistant. Answer ONLY using the facts in the context; "
            f"cite the official source. Be concise (2-3 sentences).\n\nQuestion: {q}\n\nContext:\n{ctx}\n\nAnswer:")
    out=pipe([{"role":"user","content":prompt}],max_new_tokens=200,do_sample=False)
    gen=out[0]["generated_text"]
    return gen[-1]["content"] if isinstance(gen,list) else str(gen)

def agg(ss,name):
    n=len(ss)
    if n==0: return [name,0,1.0,0.0,0.0,0.0]
    return [name,n,round(sum(s["num_faithful"] for s in ss)/n,3),
            round(sum(s["halluc"] for s in ss)/n,2),
            round(sum(s["unsupported"] for s in ss)/n,2),
            round(sum(s["provenance"] for s in ss)/n,2)]

base   = build_cases()
scaled = build_scaled_cases(SCALED_TTL)
print(f"base worked cases: {len(base)} | scaled cohort: {len(scaled)} ({sum(c['ftype']=='relative-outperformer' for c in scaled)} CQ3 + {sum(c['ftype']=='fundamentals-market-divergence' for c in scaled)} CQ1)")

PERCASE=[]  # per-case citation flags -> supports a paired McNemar test across models
def run(cohort, tag):
    out=[]
    for i,c in enumerate(cohort,1):
        a=llm_answer(c["question"],c["context"]); s=score(a,c["factnums"],c["prov_tokens"]); out.append(s)
        PERCASE.append({"model":MODEL,"cohort":tag,"label":c["label"],"cq":c["ftype"],
                        "provenance":s["provenance"],"num_faithful":s["num_faithful"],"halluc":s["halluc"]})
        if i<=6 or i%10==0:
            print(f"[{tag} {i}/{len(cohort)}] {c['label']}: faithful={s['num_faithful']} halluc={s['halluc']} unsup={s['unsupported']} prov={s['provenance']}")
    return out

onto_all = [score(c["onto_answer"],c["factnums"],c["prov_tokens"]) for c in base+scaled]
live_base   = run(base,"base")
live_scaled = run(scaled,"scaled")

rows=[agg(onto_all,"OntoKG-EQ (provenance-grounded)"),
      agg(live_base,  f"GraphRAG/LLM base [{MODEL}]"),
      agg(live_scaled,f"GraphRAG/LLM scaled-64 [{MODEL}]")]

with open(os.path.join(OUT,"graphrag_faithfulness_live.csv"),"w",newline="") as f:
    w=csv.writer(f); w.writerow(["method","cases","numeric_faithfulness","hallucinated_numbers","unsupported_assertions","provenance_coverage"])
    for r in rows: w.writerow(r)
md="# Live GraphRAG/LLM faithfulness vs OntoKG-EQ (base + scaled-64 cohort)\n\n"
md+="| Method | cases | numeric faithfulness | halluc. numbers | unsupported assertions | provenance |\n|---|---:|---:|---:|---:|---:|\n"
for r in rows: md+=f"| {r[0]} | {r[1]} | {r[2]:.2f} | {r[3]:.2f} | {r[4]:.2f} | {r[5]:.2f} |\n"
open(os.path.join(OUT,"graphrag_faithfulness_live.md"),"w").write(md)
with open(os.path.join(OUT,"graphrag_faithfulness_percase.csv"),"w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=["model","cohort","label","cq","provenance","num_faithful","halluc"]); w.writeheader()
    for r in PERCASE: w.writerow(r)
print("\n"+md)
print("per-case flags saved to /kaggle/working/graphrag_faithfulness_percase.csv")
print("Saved to /kaggle/working/graphrag_faithfulness_live.{md,csv}")
